In [2]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main/code
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [4]:

import torch
from torch import nn

class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=32):
        super(MatrixFactorization, self).__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([0.0]))

        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_ids, item_ids):
        user_vec = self.user_emb(user_ids)
        item_vec = self.item_emb(item_ids)
        dot = (user_vec * item_vec).sum(1, keepdim=True)
        return (dot + self.user_bias(user_ids) + self.item_bias(item_ids) + self.global_bias).squeeze()


In [6]:
import mlflow
import pandas as pd
from rich import print
from sklearn.model_selection import train_test_split
from torch.optim import SGD

from src.data_utils import DataName, create_dataloader, read_in_data
from src.models.MatrixFactorization import MF
from src.training.centralized import centralized_train_loop, centralized_validate_loop
from src.training.train_utils import EarlyStopper



In [7]:
import numpy as np
import torch
from torch import nn
from tqdm.auto import tqdm

In [10]:
def centralized_validate_loop(model, val_loader, optimizer, scheduler=None):
    tbar = val_loader
    loss_fn = nn.MSELoss(reduction="sum")
    model.eval()
    total_obs = 0
    total_sum_loss = 0
    with torch.no_grad():
        for idx, (inputs, target) in enumerate(tbar):
            if inputs.ndim == 3:
                inputs = inputs.squeeze(0)
                target = target.squeeze(0)
            n_obs = inputs.shape[0]
            score = model(inputs[:, 0], inputs[:, 1])

            sum_loss = loss_fn(score, target.float()).detach().numpy()
            total_sum_loss += sum_loss
            total_obs += n_obs

        avg_loss = np.sqrt(total_sum_loss / total_obs)
        if scheduler is not None:
            scheduler.step(avg_loss)

    return avg_loss, total_obs 


def centralized_train_loop(model, train_loader, optimizer, progress_bar=True) -> float:
    loss_fn = nn.MSELoss(reduction="mean")
    model.train()
    total_n_obs = 0
    total_sum_loss = 0
    tbar = tqdm(train_loader) if progress_bar else train_loader
    ten_percent = len(train_loader) // 10
    for idx, (inputs, target) in enumerate(tbar):
        if inputs.ndim == 3:
            inputs = inputs.squeeze(0)
            target = target.squeeze(0)
        n_obs = inputs.shape[0]
        optimizer.zero_grad()
        score = model(inputs[:, 0], inputs[:, 1])
        loss = loss_fn(score, target.float())
        loss.backward()  # Calculate Gradients
        optimizer.step()  # Current user's update

        total_n_obs += n_obs
        total_sum_loss += loss.detach().numpy() * n_obs
        avg_mse_loss = total_sum_loss / total_n_obs
        if (ten_percent > 0) and (idx % ten_percent == 0):
            if progress_bar:
                tbar.set_description(
                    f"Average Training Loss: {np.sqrt(avg_mse_loss):.04f} | Loss: {loss.detach():.04f}"
                )
            if np.isnan(avg_mse_loss):
                print("Training Loss is NaN")
                raise ValueError
        # if idx % 1000 == 0:
        #     tbar.set_description(f"Training Loss: {np.sqrt(avg_mse_loss):.05f} ")
    return np.sqrt(total_sum_loss / total_n_obs), total_n_obs


In [12]:
DATA_DIR = Path("/Users/haowen/Documents/Decentral RS/fed-learning-main/dataset")

In [48]:
train_df = pd.read_csv("dataset/ml100k_train_seed10.csv")
test_df = pd.read_csv("dataset/ml100k_test_seed10.csv")
n_users, n_items, _ = train_df.nunique()
n_users
train_dl = create_dataloader(train_df, dl_type="centralized", batch_size=10)
#val_dl = create_dataloader(val_df, dl_type="centralized", batch_size=10)
test_dl = create_dataloader(test_df, dl_type="centralized", batch_size=10)


In [50]:
model = MF(n_users=n_users, n_items=n_items, n_factors=30)

In [52]:
# optimizer = AdamW(model.parameters(), lr=1e-3)
optimizer = SGD(model.parameters(), lr=1e-2, weight_decay=.01, momentum=.1)
early_stopper = EarlyStopper()

In [54]:
centralized_validate_loop(model, val_dl, optimizer)

(6.708132760226375, 4800)

In [56]:
batch_size=  10
n_factors=30
lr=1e-2
weight_decay=.002
momentum=.01
sparse = False

In [158]:
train_df, val_df = train_test_split(
    train_df, test_size=0.2, random_state=0
)

train_dl = create_dataloader(train_df, dl_type="centralized", batch_size=10)
val_dl = create_dataloader(val_df, dl_type="centralized", batch_size=10)
test_dl = create_dataloader(test_df, dl_type="centralized", batch_size=10)
model = MF(n_users=n_users, n_items=n_items, n_factors=n_factors, sparse=sparse)
# optimizer = AdamW(model.parameters(), lr=1e-4)
optimizer = SGD(model.parameters(), lr=lr, weight_decay=wd, momentum=mom)
early_stopper = EarlyStopper()
val_losses = []
train_losses = []
communication_cost = 0


In [161]:
centralized_validate_loop(model, val_dl, optimizer)

(6.750496250607487, 12000)

In [58]:
tbar = val_dl
loss_fn = nn.MSELoss(reduction="sum")
model.eval()
total_obs = 0
total_sum_loss = 0

In [60]:
n_epochs = 20
val_losses = []
for epoch in range(n_epochs):
    train_loss = centralized_train_loop(model, train_dl, optimizer)
    val_loss, val_commute = centralized_validate_loop(model, val_dl, optimizer)
    print(f"Epoch: {epoch+1}, Val loss: {val_loss}")
    if early_stopper.early_stop(val_loss):
        print("Stopping Early.")
        break
    val_losses.append(np.sqrt(val_loss))

  0%|          | 0/7500 [00:00<?, ?it/s]

Epoch: 1, Val loss: 2.746220176141559

  0%|          | 0/7500 [00:00<?, ?it/s]

Epoch: 2, Val loss: 2.4291839703411964

  0%|          | 0/7500 [00:00<?, ?it/s]

...

  0%|          | 0/7500 [00:00<?, ?it/s]

Epoch: 6, Val loss: 2.367205291614041

  0%|          | 0/7500 [00:00<?, ?it/s]

Epoch: 7, Val loss: 2.3678988338273084

Stopping Early.

In [ ]:
9430